# Ladybug nightly + Icebug: Star Wars graph analytics

This notebook uses the `ladybug==0.17.0` release to query Arrow-backed graph data with Cypher, persists that graph into a local Ladybug database, reopens it read-only, exports Arrow CSR columns from disk, and hands those Arrow arrays to Icebug for Leiden community detection.

The dataset is intentionally small and friendly: a Star Wars interaction graph with characters, factions, homeworlds, and weighted relationships.

## Environment

Dependencies are declared in `pyproject.toml`. From a fresh checkout, start Jupyter with:

```bash
uv run jupyter notebook ladybug_icebug_starwars.ipynb
```

Or execute the notebook from the CLI with:

```bash
uv run jupyter nbconvert --to notebook --execute ladybug_icebug_starwars.ipynb --inplace
```

In [1]:
# Optional setup cell: sync dependencies from pyproject.toml.
# If you launched with `uv run jupyter notebook`, this is already done.
!uv sync

Resolved 105 packages in 2ms
Checked 101 packages in 1ms


In [2]:
from pathlib import Path
import time

import ladybug
import icebug as ib
import pandas as pd
import pyarrow as pa

print("ladybug", ladybug.__version__)
print("icebug", ib.__version__)

ladybug 0.17.0
icebug 12.8


## Build a friendly Arrow graph

Ladybug can register Arrow tables directly as graph node and relationship tables. Relationship tables use endpoint columns named `from` and `to`, with values pointing at node row ids.

In [3]:
characters = pa.table({
    "id": pa.array(list(range(18)), type=pa.int64()),
    "name": [
        "Luke Skywalker", "Leia Organa", "Han Solo", "Chewbacca", "R2-D2", "C-3PO",
        "Obi-Wan Kenobi", "Yoda", "Darth Vader", "Emperor Palpatine", "Tarkin",
        "Boba Fett", "Lando Calrissian", "Jabba the Hutt", "Mon Mothma",
        "Wedge Antilles", "Mara Jade", "Ahsoka Tano",
    ],
    "faction": [
        "Rebellion", "Rebellion", "Rebellion", "Rebellion", "Rebellion", "Rebellion",
        "Jedi", "Jedi", "Empire", "Empire", "Empire", "Underworld", "Rebellion",
        "Underworld", "Rebellion", "Rebellion", "Underworld", "Jedi",
    ],
    "homeworld": [
        "Tatooine", "Alderaan", "Corellia", "Kashyyyk", "Naboo", "Tatooine",
        "Stewjon", "Dagobah", "Tatooine", "Naboo", "Eriadu", "Kamino", "Socorro",
        "Nal Hutta", "Chandrila", "Corellia", "Coruscant", "Shili",
    ],
    "role": [
        "Jedi hopeful", "General", "Smuggler", "Co-pilot", "Astromech", "Protocol droid",
        "Mentor", "Grand master", "Sith lord", "Emperor", "Grand Moff", "Bounty hunter",
        "Administrator", "Crime lord", "Chancellor", "Pilot", "Agent", "Fulcrum",
    ],
})

name_to_id = dict(zip(characters["name"].to_pylist(), characters["id"].to_pylist()))

interaction_names = pa.table({
    "source": [
        "Luke Skywalker", "Luke Skywalker", "Han Solo", "Leia Organa", "R2-D2", "R2-D2",
        "C-3PO", "Obi-Wan Kenobi", "Yoda", "Obi-Wan Kenobi", "Ahsoka Tano", "Ahsoka Tano",
        "Darth Vader", "Darth Vader", "Tarkin", "Darth Vader", "Darth Vader", "Darth Vader",
        "Boba Fett", "Boba Fett", "Jabba the Hutt", "Jabba the Hutt", "Lando Calrissian",
        "Lando Calrissian", "Mon Mothma", "Mon Mothma", "Wedge Antilles", "Wedge Antilles",
        "Mara Jade", "Mara Jade", "Mara Jade",
    ],
    "target": [
        "Leia Organa", "Han Solo", "Chewbacca", "Han Solo", "C-3PO", "Luke Skywalker",
        "Leia Organa", "Luke Skywalker", "Luke Skywalker", "Yoda", "Obi-Wan Kenobi", "Yoda",
        "Emperor Palpatine", "Tarkin", "Emperor Palpatine", "Luke Skywalker", "Leia Organa",
        "Obi-Wan Kenobi", "Darth Vader", "Jabba the Hutt", "Han Solo", "Leia Organa",
        "Han Solo", "Leia Organa", "Leia Organa", "Wedge Antilles", "Luke Skywalker",
        "Lando Calrissian", "Emperor Palpatine", "Luke Skywalker", "Boba Fett",
    ],
    "scene": [
        "twins and rebellion", "rescues and raids", "lifelong crew", "command and chemistry",
        "droid duo", "secret plans", "translation and diplomacy", "training", "training",
        "jedi council", "clone wars", "jedi order", "master and apprentice", "imperial command",
        "death star politics", "family conflict", "imperial pursuit", "old master", "bounty contract",
        "underworld contract", "debt", "palace rescue", "old friends", "alliance planning",
        "rebel leadership", "fleet command", "rogue squadron", "battle of endor", "emperor's hand",
        "rivalry to trust", "underworld overlap",
    ],
    "strength": pa.array([8, 7, 10, 8, 9, 7, 5, 9, 9, 6, 6, 5, 10, 7, 6, 10, 6, 9, 6, 8, 8, 4, 8, 6, 8, 5, 7, 6, 7, 6, 4], type=pa.int64()),
})

interactions = pa.table({
    "from": pa.array([name_to_id[name] for name in interaction_names["source"].to_pylist()], type=pa.int64()),
    "to": pa.array([name_to_id[name] for name in interaction_names["target"].to_pylist()], type=pa.int64()),
    "scene": interaction_names["scene"],
    "strength": interaction_names["strength"],
})

characters, interactions

(pyarrow.Table
 id: int64
 name: string
 faction: string
 homeworld: string
 role: string
 ----
 id: [[0,1,2,3,4,...,13,14,15,16,17]]
 name: [["Luke Skywalker","Leia Organa","Han Solo","Chewbacca","R2-D2",...,"Jabba the Hutt","Mon Mothma","Wedge Antilles","Mara Jade","Ahsoka Tano"]]
 faction: [["Rebellion","Rebellion","Rebellion","Rebellion","Rebellion",...,"Underworld","Rebellion","Rebellion","Underworld","Jedi"]]
 homeworld: [["Tatooine","Alderaan","Corellia","Kashyyyk","Naboo",...,"Nal Hutta","Chandrila","Corellia","Coruscant","Shili"]]
 role: [["Jedi hopeful","General","Smuggler","Co-pilot","Astromech",...,"Crime lord","Chancellor","Pilot","Agent","Fulcrum"]],
 pyarrow.Table
 from: int64
 to: int64
 scene: string
 strength: int64
 ----
 from: [[0,0,2,1,4,...,15,15,16,16,16]]
 to: [[1,2,3,2,5,...,0,12,9,0,11]]
 scene: [["twins and rebellion","rescues and raids","lifelong crew","command and chemistry","droid duo",...,"rogue squadron","battle of endor","emperor's hand","rivalry to tru

## Query the in-memory Arrow graph with Cypher

In [4]:
mem_db = ladybug.Database()
mem = ladybug.Connection(mem_db)
mem.create_arrow_table("Character", characters)
mem.create_arrow_rel_table("INTERACTS_WITH", interactions, "Character", "Character")

mem.query_as_arrow("""
MATCH (a:Character)-[r:INTERACTS_WITH]->(b:Character)
RETURN a.name AS source, a.faction AS source_faction,
       b.name AS target, b.faction AS target_faction,
       r.scene AS scene, r.strength AS strength
ORDER BY r.strength DESC, source
LIMIT 12
""", 4096).get_as_arrow().to_pandas()

,source,source_faction,target,target_faction,scene,strength
0,Darth Vader,Empire,Emperor Palpatine,Empire,master and apprentice,10
1,Darth Vader,Empire,Luke Skywalker,Rebellion,family conflict,10
2,Han Solo,Rebellion,Chewbacca,Rebellion,lifelong crew,10
3,Darth Vader,Empire,Obi-Wan Kenobi,Jedi,old master,9
4,Obi-Wan Kenobi,Jedi,Luke Skywalker,Rebellion,training,9
5,R2-D2,Rebellion,C-3PO,Rebellion,droid duo,9
6,Yoda,Jedi,Luke Skywalker,Rebellion,training,9
7,Boba Fett,Underworld,Jabba the Hutt,Underworld,underworld contract,8
8,Jabba the Hutt,Underworld,Han Solo,Rebellion,debt,8
9,Lando Calrissian,Rebellion,Han Solo,Rebellion,old friends,8


In [5]:
mem.query_as_arrow("""
MATCH (a:Character)-[r:INTERACTS_WITH]->(b:Character)
RETURN a.faction AS faction, count(r) AS outgoing_interactions
ORDER BY outgoing_interactions DESC
""", 4096).get_as_arrow().to_pandas()

,faction,outgoing_interactions
0,Rebellion,13
1,Underworld,7
2,Empire,6
3,Jedi,5


## Persist Arrow data to native Ladybug tables

The in-memory Arrow tables are useful for exploration, but `create_arrow_table` keeps data Arrow-backed. For the disk-backed demo, this section registers Arrow staging tables, creates native node and relationship tables, then uses Cypher `COPY ... FROM (MATCH ... RETURN ...)` to load native storage.

For Leiden, the persisted `INTERACTS_WITH` table stores both directions of each interaction so the native CSR represents an undirected graph.

In [6]:
db_path = Path("starwars_alliances.lbdb")
if db_path.exists():
    db_path.unlink()

native_interactions = pa.concat_tables([
    interactions,
    pa.table({
        "from": interactions["to"],
        "to": interactions["from"],
        "scene": interactions["scene"],
        "strength": interactions["strength"],
    }),
])

disk = ladybug.Connection(ladybug.Database(db_path))
disk.create_arrow_table("CharacterArrow", characters)
disk.create_arrow_table("InteractsArrow", native_interactions)
disk.execute("""
CREATE NODE TABLE Character(
    id INT64,
    name STRING,
    faction STRING,
    homeworld STRING,
    role STRING,
    PRIMARY KEY(id)
)
""")
disk.execute("""
CREATE REL TABLE INTERACTS_WITH(
    FROM Character TO Character,
    scene STRING,
    strength INT64
)
""")
disk.execute("""
COPY Character FROM (
    MATCH (c:CharacterArrow)
    RETURN c.id, c.name, c.faction, c.homeworld, c.role
)
""")
disk.execute("""
COPY INTERACTS_WITH FROM (
    MATCH (e:InteractsArrow)
    RETURN e.from, e.to, e.scene, e.strength
)
""")
disk.drop_arrow_table("CharacterArrow")
disk.drop_arrow_table("InteractsArrow")
disk.close()

db_path, db_path.stat().st_size, characters.num_rows, native_interactions.num_rows

(PosixPath('starwars_alliances.lbdb'), 331776, 18, 62)

## Read native relationship CSR from disk, then run Icebug

The CSR exporter expects a rowid-only relationship projection: source node row id, relationship row id, and destination node row id. Relationship properties can be queried separately, but `.csr()` itself only consumes the rowid projection and returns Arrow arrays for `indptr`, `indices`, and `edge_ids` without materializing separate CSR tables.

In [7]:
readonly = ladybug.Connection(ladybug.Database(db_path, read_only=True))

n_nodes = readonly.query_as_arrow(
    "MATCH (c:Character) RETURN count(c) AS n",
    1024,
).get_as_arrow().column(0)[0].as_py()

relationship_result = readonly.query_as_arrow("""
MATCH (a)-[b:INTERACTS_WITH]->(c)
RETURN a.rowid AS src_rowid,
       b.rowid AS rel_rowid,
       c.rowid AS dst_rowid
""", 4096)
relationship_csr = relationship_result.csr()

indptr_arrow = relationship_csr.indptr.cast(pa.uint64())
indices_arrow = relationship_csr.indices.cast(pa.uint64())
edge_ids_arrow = relationship_csr.edge_ids.cast(pa.uint64())
characters_from_disk = readonly.query_as_arrow(
    "MATCH (c:Character) RETURN c.rowid AS rowid, c.id AS id, c.name AS name, c.faction AS faction ORDER BY c.rowid",
    4096,
).get_as_arrow().to_pandas()
relationship_properties = readonly.query_as_arrow(
    "MATCH ()-[r:INTERACTS_WITH]->() RETURN r.rowid AS rowid, r.scene AS scene, r.strength AS strength ORDER BY r.rowid",
    4096,
).get_as_arrow().to_pandas()

_arrow_registry = {
    "indptr": indptr_arrow,
    "indices": indices_arrow,
    "edge_ids": edge_ids_arrow,
    "characters": characters_from_disk,
    "relationship_properties": relationship_properties,
}

t0 = time.time()
graph = ib.Graph.fromCSR(n_nodes, False, indices_arrow, indptr_arrow)
build_elapsed = time.time() - t0
print(f"Icebug graph: {graph.numberOfNodes()} nodes, {graph.numberOfEdges()} undirected edges; built in {build_elapsed:.4f}s")

Icebug graph: 18 nodes, 31 undirected edges; built in 0.0001s


In [8]:
t0 = time.time()
leiden = ib.community.ParallelLeidenView(graph, iterations=4, gamma=1.0, randomize=False)
leiden.run()
partition = leiden.getPartition()
elapsed = time.time() - t0
modularity = ib.community.Modularity().getQuality(partition, graph)

communities = characters_from_disk.copy()
communities["community"] = [partition.subsetOf(i) for i in range(graph.numberOfNodes())]
communities = communities.sort_values(["community", "faction", "name"]).reset_index(drop=True)

print(f"Leiden found {partition.numberOfSubsets()} communities in {elapsed:.4f}s; modularity Q={modularity:.4f}")
communities

Leiden found 4 communities in 0.0060s; modularity Q=0.3730


,rowid,id,name,faction,community
0,17,17,Ahsoka Tano,Jedi,0
1,6,6,Obi-Wan Kenobi,Jedi,0
2,7,7,Yoda,Jedi,0
3,0,0,Luke Skywalker,Rebellion,0
4,3,3,Chewbacca,Rebellion,1
5,2,2,Han Solo,Rebellion,1
6,12,12,Lando Calrissian,Rebellion,1
7,1,1,Leia Organa,Rebellion,1
8,14,14,Mon Mothma,Rebellion,1
9,15,15,Wedge Antilles,Rebellion,1


In [9]:
summary = (
    communities.groupby(["community", "faction"])
    .size()
    .rename("characters")
    .reset_index()
    .sort_values(["community", "characters"], ascending=[True, False])
)
summary

,community,faction,characters
0,0,Jedi,3
1,0,Rebellion,1
2,1,Rebellion,6
3,1,Underworld,1
4,2,Rebellion,2
5,3,Empire,3
6,3,Underworld,2


## What happened

1. The Star Wars graph started as ordinary PyArrow tables.
2. Ladybug queried those Arrow tables in memory using Cypher.
3. The Arrow tables were loaded into native `Character` and `INTERACTS_WITH` tables with Cypher `COPY`.
4. A read-only Ladybug connection queried the native relationships and exported CSR with `query_as_arrow(...).csr()`.
5. Icebug consumed those Arrow CSR arrays directly with `Graph.fromCSR` and ran Leiden.